# NumPy for Images

> **Beginner · Foundation**


## Why this matters

OpenCV images are NumPy arrays. Understanding shape, dtype, slicing, and vectorization prevents most early image-processing mistakes.

**Where it appears:** Cropping, masking, alpha compositing, batch transforms, and fast pixel-wise operations.


## Learning Objectives

- Understand that an OpenCV image IS a NumPy array (shape, dtype, strides)
- Use slicing for regions of interest instead of manual pixel loops
- Use broadcasting and vectorized ops for pixel-level transforms


## Prerequisites

01 Python Foundations

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`shape`, `dtype`, slicing, boolean masks, broadcasting, `np.clip`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### NumPy for Computer Vision

OpenCV images are plain `numpy.ndarray` objects with shape `(height, width,
channels)` and dtype `uint8` for typical 8-bit images. Almost every
performance and correctness bug in OpenCV code traces back to a NumPy
misunderstanding: wrong axis order, dtype overflow, or an accidental loop
where a vectorized operation should be used. This notebook builds that
foundation before any actual OpenCV function is introduced.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


### 1. Shape, dtype, and what they mean for images

A color image's shape is `(rows, cols, channels)` -- note **height first**, which trips up almost everyone coming from (x, y) thinking.


In [ ]:
def describe_array(name: str, arr) -> None:
    """Print the properties that matter most when debugging image code."""
    print(
        f"{name}: shape={arr.shape}, dtype={arr.dtype}, "
        f"min={arr.min()}, max={arr.max()}, size_bytes={arr.nbytes}"
    )


# Load a real landscape image instead of np.zeros
color_image = load_real_image("images/landscapes", "mountain.jpg")
# Convert to grayscale (we will learn this formally later, but it drops the channel dimension)
gray_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2GRAY)

describe_array("color_image", color_image)
describe_array("gray_image", gray_image)
show_grid([("Color (3D)", color_image), ("Grayscale (2D)", gray_image)], cols=2)

### 2. Slicing as Region of Interest (ROI)

Cropping in OpenCV is just NumPy slicing: `image[y1:y2, x1:x2]`. This returns a *view*, not a copy -- editing the slice edits the original array.


In [ ]:
def extract_roi(image: np.ndarray, y1: int, y2: int, x1: int, x2: int) -> np.ndarray:
    """Return a view into `image` for the given rectangle (row-major slicing)."""
    return image[y1:y2, x1:x2]


# Work on a copy of the real image so we don't permanently ruin the loaded one
canvas = color_image.copy()
roi = extract_roi(canvas, 50, 150, 50, 150)
roi[:] = (0, 255, 0)  # modifies canvas too, because roi is a VIEW not a copy

print("Notice the bright green square where we modified the ROI view.")
show(canvas, "Canvas with modified ROI view")

# Force an independent copy when you don't want to mutate the source:
roi_copy = extract_roi(canvas, 0, 50, 0, 50).copy()
roi_copy[:] = (255, 0, 0)
print("Canvas unaffected by editing roi_copy (top-left is NOT red).")

### 3. Dtype overflow and vectorized math

`uint8` wraps around instead of raising an error on overflow. Arithmetic on images almost always needs an intermediate signed/float dtype, then a clip back to `uint8` -- this exact pattern reappears in brightness/contrast and noise-related notebooks.


In [ ]:
def adjust_brightness(image: np.ndarray, delta: int) -> np.ndarray:
    """Add `delta` to every pixel safely (no wraparound), vectorized (no Python loop)."""
    return np.clip(image.astype(np.int16) + delta, 0, 255).astype(np.uint8)


sample = np.array([[250, 10], [128, 5]], dtype=np.uint8)
wrong = sample + np.uint8(20)  # silently wraps around (bug!)
correct = adjust_brightness(sample, 20)

print("uint8 overflow (buggy):\n", wrong)
print("clipped correctly:\n", correct)

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — NumPy for Computer Vision: Green Screen Compositing (Chroma Keying)

Chroma keying is a classic computer vision technique where a specific color range (typically green) is masked out and replaced with a background image. By leveraging vectorized NumPy operations, boolean indexing, and broadcasting, we perform this operation instantly without any slow nested Python loops.


In [ ]:
# Create a synthetic green screen foreground (green background with a red object)
height, width = 240, 320
fg = np.zeros((height, width, 3), dtype=np.uint8)
fg[:, :] = [0, 255, 0]  # Fill with BGR green
fg[60:180, 80:240] = [0, 0, 255]  # Red foreground rectangle

# Use our real mountain image as the background!
bg_real = load_real_image("images/landscapes", "mountain.jpg")
bg = cv2.resize(bg_real, (width, height))

# Create a boolean mask where the color is green
# BGR green is [0, 255, 0]. We select pixels where green is high, blue/red are low
is_green = (fg[:, :, 1] > 200) & (fg[:, :, 0] < 50) & (fg[:, :, 2] < 50)

# Expand the 2D boolean mask to 3D to match (height, width, channels)
mask_3d = np.expand_dims(is_green, axis=2)

# Vectorized compositing: replace green pixels with background, keep foreground elsewhere
composited = np.where(mask_3d, bg, fg)

show_grid([("Foreground", fg), ("Background", bg), ("Composited", composited)])

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — NumPy for Computer Vision
1. Write `extract_channel(image, index)` returning a single-channel 2D array via slicing.
2. Write `blend(a, b, alpha)` that vectorized-blends two same-shape float arrays and returns `uint8`.
3. Explain, in a markdown cell, why `image[:, :, ::-1]` converts BGR to RGB without calling any OpenCV function.

Use the empty cell below to work through them.


#### Solutions — NumPy for Computer Vision

In [ ]:
# Solution 1: extract_channel returning a single-channel 2D array
def extract_channel(image: np.ndarray, index: int) -> np.ndarray:
    """Extract a single color channel from a BGR image using slicing."""
    return image[:, :, index]

In [ ]:
# Solution 2: blend two float arrays and return uint8
def blend(a: np.ndarray, b: np.ndarray, alpha: float) -> np.ndarray:
    """Blend two images of the same shape using vectorized linear combination."""
    blended = (1.0 - alpha) * a.astype(np.float32) + alpha * b.astype(np.float32)
    return np.clip(blended, 0, 255).astype(np.uint8)


if "fg" in locals() and "bg" in locals():
    show(blend(fg, bg, 0.5), "Blended (50% alpha)")

In [ ]:
# Solution 3: Explain image[:, :, ::-1] channel reversal
# In OpenCV, color images are stored as NumPy arrays in BGR channel order.
# The slice syntax `image[:, :, ::-1]` operates on the third axis (channels).
# The `::-1` slice step reverses the order of elements along this axis, changing
# BGR (0, 1, 2) to RGB (2, 1, 0). Because slicing in NumPy returns a 'view' (alias)
# of the data instead of copying it, this conversion is extremely fast and O(1) in memory.

## Summary

You can inspect and manipulate image arrays without slow Python pixel loops or accidental integer overflow.

- **Best Practices:** Check `shape` and `dtype` before processing, vectorize operations, and make copies when an operation must not modify the original array.
- **Common Pitfalls:** Assuming an image is RGB, overflowing `uint8` values, and unintentionally editing a view returned by a slice.